In [ ]:
import sys

sys.path.append("../src/")

In [ ]:
from error_analysis import (
    set_aws_credentials,
    set_mlflow_env,
    get_file_system,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
set_aws_credentials()
set_mlflow_env()

fasttext_run_id = "3927ef0bee3b4be79117cb73f02ca376"
camembert_run_id = "46998e8160ad4b33b494f31f339fc1ae"

fs = get_file_system()
s3_path = f"projet-ape/estoril/predictions_{fasttext_run_id}_{camembert_run_id}.csv"

In [ ]:
import pandas as pd

In [ ]:
with fs.open(s3_path) as f:
    df_test = pd.read_csv(f)

In [ ]:
# Compute accuracy
fasttext_accuracy = accuracy_score(df_test["y_true"], df_test["y_pred_fasttext"])
camembert_accuracy = accuracy_score(df_test["y_true"], df_test["y_pred_camembert"])
print(f"FastText accuracy: {fasttext_accuracy}")
print(f"Camembert accuracy: {camembert_accuracy}")

# Compute micro-averaged precision, recall and F1
fasttext_precision = precision_score(df_test["y_true"], df_test["y_pred_fasttext"], average="micro")
fasttext_recall = recall_score(df_test["y_true"], df_test["y_pred_fasttext"], average="micro")
fasttext_f1 = f1_score(df_test["y_true"], df_test["y_pred_fasttext"], average="micro")
print(f"FastText micro-averaged precision: {fasttext_precision}")
print(f"FastText micro-averaged recall: {fasttext_recall}")
print(f"FastText micro-averaged F1: {fasttext_f1}")

camembert_precision = precision_score(
    df_test["y_true"], df_test["y_pred_camembert"], average="micro"
)
camembert_recall = recall_score(df_test["y_true"], df_test["y_pred_camembert"], average="micro")
camembert_f1 = f1_score(df_test["y_true"], df_test["y_pred_camembert"], average="micro")
print(f"Camembert micro-averaged precision: {camembert_precision}")
print(f"Camembert micro-averaged recall: {camembert_recall}")
print(f"Camembert micro-averaged F1: {camembert_f1}")

# Compute macro-averaged precision, recall and F1
fasttext_precision = precision_score(df_test["y_true"], df_test["y_pred_fasttext"], average="macro")
fasttext_recall = recall_score(df_test["y_true"], df_test["y_pred_fasttext"], average="macro")
fasttext_f1 = f1_score(df_test["y_true"], df_test["y_pred_fasttext"], average="macro")
print(f"FastText macro-averaged precision: {fasttext_precision}")
print(f"FastText macro-averaged recall: {fasttext_recall}")
print(f"FastText macro-averaged F1: {fasttext_f1}")

camembert_precision = precision_score(
    df_test["y_true"], df_test["y_pred_camembert"], average="macro"
)
camembert_recall = recall_score(df_test["y_true"], df_test["y_pred_camembert"], average="macro")
camembert_f1 = f1_score(df_test["y_true"], df_test["y_pred_camembert"], average="macro")
print(f"Camembert macro-averaged precision: {camembert_precision}")
print(f"Camembert macro-averaged recall: {camembert_recall}")
print(f"Camembert macro-averaged F1: {camembert_f1}")

In [ ]:
df_test.head()

In [ ]:
df_test["all_agree"] = (df_test["y_pred_fasttext"] == df_test["y_pred_camembert"]) & (
    df_test["y_pred_camembert"] == df_test["y_true"]
)
df_test["fasttext_agrees"] = (df_test["y_pred_fasttext"] == df_test["y_true"]) & (
    df_test["y_pred_fasttext"] != df_test["y_pred_camembert"]
)
df_test["camembert_agrees"] = (df_test["y_pred_camembert"] == df_test["y_true"]) & (
    df_test["y_pred_fasttext"] != df_test["y_pred_camembert"]
)
df_test["camembert_fasttext_agree"] = (
    df_test["y_pred_fasttext"] == df_test["y_pred_camembert"]
) & (df_test["y_pred_fasttext"] != df_test["y_true"])

In [ ]:
df_test.head()

In [ ]:
print(f'Rate of agreement of all: {df_test["all_agree"].mean()}')
print(
    f'Rate at which fastText agrees with annotation but not with Camembert: {df_test["fasttext_agrees"].mean()}'
)
print(
    f'Rate at which Camembert agrees with annotation but not with fastText: {df_test["camembert_agrees"].mean()}'
)
print(
    f'Rate at which Camembert agrees with fastText but not with annotation: {df_test["camembert_fasttext_agree"].mean()}'
)

In [ ]:
pd.set_option("display.max_columns", None)  # or 1000
pd.set_option("display.max_rows", None)  # or 1000
pd.set_option("display.max_colwidth", None)  # or 199

In [ ]:
df_test["y_true_div"] = df_test["y_true"].str[:2]
df_test["count"] = 1
division_stats = df_test.groupby("y_true_div").agg(
    {
        "all_agree": "mean",
        "fasttext_agrees": "mean",
        "camembert_agrees": "mean",
        "camembert_fasttext_agree": "mean",
        "count": "sum",
    }
)
division_stats

In [ ]:
division_stats.sort_values("fasttext_agrees", ascending=False).head(10)

In [ ]:
with fs.open("projet-ape/data/naf_2008_div.csv", "r") as f:
    naf_div_codes = pd.read_csv(f, sep=";")
naf_div_codes["code"] = naf_div_codes["code"].astype(str).str.zfill(2)
naf_div_codes

In [ ]:
division_stats.sort_values("fasttext_agrees", ascending=False).head(10).merge(
    naf_div_codes, left_index=True, right_on="code"
)

In [ ]:
division_stats.sort_values("camembert_agrees", ascending=False).head(10).merge(
    naf_div_codes, left_index=True, right_on="code"
)

In [ ]:
division_stats.sort_values("camembert_fasttext_agree", ascending=False).head(10).merge(
    naf_div_codes, left_index=True, right_on="code"
)

In [ ]:
with fs.open("projet-ape/data/naf2008.csv", "r") as f:
    naf_codes = pd.read_csv(f, sep=";")

df_test = df_test.merge(
    naf_codes.rename(columns={"libelle": "libelle_annotation"}),
    how="inner",
    left_on="y_true",
    right_on="code",
)
df_test = df_test.merge(
    naf_codes.rename(columns={"libelle": "libelle_pred"}),
    how="inner",
    left_on="y_pred_camembert",
    right_on="code",
)

In [ ]:
df_test.columns

In [ ]:
df_test[df_test["fasttext_agrees"]][
    [
        "libelle_activite_apet",
        "liasse_type",
        "y_true",
        "libelle_annotation",
        "y_pred_camembert",
        "libelle_pred",
    ]
].head(10)

In [ ]:
df_test.loc[df_test.camembert_fasttext_agree].to_csv("camembert_fasttext_agree.csv", index=False)